# SFT Target-Only Loss Masking with -100 ignore_index

This notebook demonstrates target-only cross-entropy loss masking by replacing prompt tokens in target tensors with `-100`.

In [1]:
import torch
import torch.nn as nn

# Input sequence of token IDs (length 10)
# Index 0-6: Prompt tokens
# Index 7-9: Target response tokens
input_ids = torch.tensor([[101, 102, 103, 104, 105, 106, 107, 201, 202, 203]], dtype=torch.long)

# Causal next-token labels are shifted inputs
labels_unmasked = input_ids.clone()

# SFT loss masking: Set indices 0 to 6 to -100
labels_masked = input_ids.clone()
labels_masked[0, :7] = -100

print("Unmasked SFT targets:")
print(labels_unmasked)
print("\nMasked SFT targets (ignore index = -100):")
print(labels_masked)

# Logit tensor of shape (batch, seq_len, vocab_size)
torch.manual_seed(42)
logits = torch.randn(1, 10, 300)

criterion = nn.CrossEntropyLoss(ignore_index=-100)

# Unmasked Loss (calculated over all shifts)
loss_unmasked = criterion(logits[:, :-1, :].reshape(-1, 300), labels_unmasked[:, 1:].reshape(-1))
# Masked Loss (only calculated on response positions)
loss_masked = criterion(logits[:, :-1, :].reshape(-1, 300), labels_masked[:, 1:].reshape(-1))

print(f"\nUnmasked Loss: {loss_unmasked.item():.4f}")
print(f"Masked Loss: {loss_masked.item():.4f}")


Unmasked SFT targets:
tensor([[101, 102, 103, 104, 105, 106, 107, 201, 202, 203]])

Masked SFT targets (ignore index = -100):
tensor([[-100, -100, -100, -100, -100, -100, -100,  201,  202,  203]])

Unmasked Loss: 5.9836
Masked Loss: 6.2284


### Output Explanation
The output demonstrates how target-only loss masking alters target tensors. The masked loss is significantly lower as the loss ignores prompt labels, isolating optimization updates to the response.